# Topic 2: Spark SQL & Views

> **Phase 3 → Week 5 → Topic 2**

---

## SQL and DataFrame API — Same Engine, Same Plan

```
DataFrame API                     Spark SQL
─────────────────────────────     ─────────────────────────────
df.groupBy("dept")                SELECT dept, AVG(salary)
  .agg(F.avg("salary"))      ≡   FROM employees
  .orderBy("dept")               GROUP BY dept
                                  ORDER BY dept

Both produce the SAME logical plan → same physical plan → same execution
```

The DataFrame API and Spark SQL are **two syntaxes for the same thing**. Catalyst compiles both to an identical logical plan. You choose based on preference — SQL is great for ad-hoc analysis, DataFrame API for programmatic pipelines.

---

## View Types

| View Type | Scope | Lifetime | Method |
|-----------|-------|---------|--------|
| Temp View | Current SparkSession | Until session ends | `createOrReplaceTempView("name")` |
| Global Temp View | ALL SparkSessions in same app | Until app ends | `createOrReplaceGlobalTempView("name")` |
| Hive Table | Permanent | Persists across sessions | `df.write.saveAsTable("name")` (requires Hive metastore) |

**Global views** are accessed with the `global_temp` prefix: `SELECT * FROM global_temp.my_view`.

---

## Interview Cheat Sheet

**Q: What's the difference between createTempView and createOrReplaceTempView?**
> `createTempView()` throws an `AnalysisException` if a view with that name already exists. `createOrReplaceTempView()` silently overwrites the existing view. In production code, always use `createOrReplaceTempView` to avoid brittle code that breaks on second run.

**Q: Does Spark SQL perform the same as the DataFrame API?**
> Yes — exactly the same. Both are compiled by Catalyst into a logical plan, which is optimized identically. The only difference is syntax. Choose SQL for readability when the query is complex; choose DataFrame API when you need to build queries programmatically.

**Q: What is a global temporary view?**
> A global temporary view is visible across all SparkSessions within the same Spark application (same JVM process). It's stored in the `global_temp` database. Access it with `SELECT * FROM global_temp.view_name`. Useful for sharing intermediate results between different sessions in the same application.

**Q: What advanced SQL features does Spark SQL support?**
> Spark SQL supports window functions (`OVER`/`PARTITION BY`), CTEs (`WITH`), subqueries, lateral views, PIVOT, UNPIVOT, ROLLUP, CUBE, GROUPING SETS, and all standard JOIN types. It also supports ANSI SQL compliance mode.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("Week5 - Spark SQL") \
    .master("local[4]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

customers = spark.read.csv("/workspace/week4/data/customers.csv", header=True, inferSchema=True)
orders    = spark.read.csv("/workspace/week4/data/orders.csv",    header=True, inferSchema=True)
products  = spark.read.csv("/workspace/week4/data/products.csv",  header=True, inferSchema=True)
sales     = spark.read.csv("/workspace/week4/data/sales.csv",     header=True, inferSchema=True)

print("Data loaded")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/13 15:38:35 WARN Utils: Your hostname, kali, resolves to a loopback address: 127.0.1.1; using 10.237.41.61 instead (on interface wlan0)
26/06/13 15:38:35 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/13 15:38:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/06/13 15:38:40 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: /workspace/week4/data/customers.csv.
java.io.FileNotFoundException: File /workspace/week4/data/customers.csv does not exist
	at org.apache.hadoop.fs.RawLocalFileSystem.deprecatedGetFileStatus(RawLocalFileSystem.java:980)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileLinkStatusInternal(RawLocalFileSystem.java:1301)
	at org.apache.hadoop.fs.RawLocalFileSystem.getFileStatus(RawLocalFileSystem.java:970)
	at org.apache.hadoop.fs.FilterFileSystem.getFileStatus(FilterFileSystem.java:462)
	at org.apache.spark.sql.execution.streaming.sinks.FileStreamSink$.hasMetadata(FileStreamSink.scala:58)
	at org.apache.spark.sql.execution.datasources.DataSource.resolveRelation(DataSource.scala:384)
	at org.apache.spark.sql.catalyst.analysis.ResolveDataSource.org$apache$spark$sql$catalyst$analysis$ResolveDataSource$$loadV1BatchSource(ResolveDataSource.scala:143)
	at org.apache.spa

AnalysisException: [PATH_NOT_FOUND] Path does not exist: file:/workspace/week4/data/customers.csv. SQLSTATE: 42K03

## Part 1: Registering Views and Running SQL

In [2]:
# Register all tables as temp views
customers.createOrReplaceTempView("customers")
orders.createOrReplaceTempView("orders")
products.createOrReplaceTempView("products")
sales.createOrReplaceTempView("sales")

# List all registered views
print("Registered temp views:")
spark.catalog.listTables()
for t in spark.catalog.listTables():
    print(f"  {t.name} ({t.tableType})")

NameError: name 'customers' is not defined

In [3]:
# Basic Spark SQL
result = spark.sql("""
    SELECT 
        c.name         AS customer_name,
        c.country,
        COUNT(o.order_id)   AS total_orders,
        SUM(o.amount)       AS total_spent,
        ROUND(AVG(o.amount), 2) AS avg_order
    FROM customers c
    LEFT JOIN orders o ON c.customer_id = o.customer_id
    GROUP BY c.name, c.country
    ORDER BY total_spent DESC NULLS LAST
""")

print("Customer spend summary (SQL):")
result.show()

{"ts": "2026-06-13 15:38:41.377", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `customers` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o30.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `customers` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABL

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `customers` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 8 pos 9;
'Sort ['total_spent DESC NULLS LAST], true
+- 'Aggregate ['c.name, 'c.country], ['c.name AS customer_name#0, 'c.country, 'COUNT('o.order_id) AS total_orders#1, 'SUM('o.amount) AS total_spent#2, 'ROUND('AVG('o.amount), 2) AS avg_order#3]
   +- 'Join LeftOuter, ('c.customer_id = 'o.customer_id)
      :- 'SubqueryAlias c
      :  +- 'UnresolvedRelation [customers], [], false
      +- 'SubqueryAlias o
         +- 'UnresolvedRelation [orders], [], false


In [4]:
# Identical query using DataFrame API — produces the SAME plan
result_df = customers.alias("c") \
    .join(orders.alias("o"), on="customer_id", how="left") \
    .groupBy("c.name", "c.country") \
    .agg(
        F.count("o.order_id").alias("total_orders"),
        F.sum("o.amount").alias("total_spent"),
        F.round(F.avg("o.amount"), 2).alias("avg_order")
    ) \
    .orderBy(F.col("total_spent").desc_nulls_last())

print("Identical result from DataFrame API:")
result_df.show()

# Verify same plan
print("\nSQL plan:")
result.explain()
print("\nDataFrame API plan:")
result_df.explain()

NameError: name 'customers' is not defined

## Part 2: Advanced SQL Features

In [5]:
# CTEs (Common Table Expressions) — WITH clause
cte_result = spark.sql("""
    WITH
    customer_spend AS (
        SELECT customer_id, SUM(amount) AS total_spent, COUNT(*) AS order_count
        FROM orders
        WHERE status = 'delivered'
        GROUP BY customer_id
    ),
    high_value AS (
        SELECT customer_id, total_spent, order_count
        FROM customer_spend
        WHERE total_spent > 500
    )
    SELECT c.name, c.country, c.tier, h.total_spent, h.order_count
    FROM customers c
    INNER JOIN high_value h ON c.customer_id = h.customer_id
    ORDER BY h.total_spent DESC
""")

print("CTEs — high value customers:")
cte_result.show()

{"ts": "2026-06-13 15:38:41.758", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `customers` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o30.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `customers` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABL

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `customers` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 15 pos 9;
'Sort ['h.total_spent DESC NULLS LAST], true
+- 'Project ['c.name, 'c.country, 'c.tier, 'h.total_spent, 'h.order_count]
   +- 'Join Inner, ('c.customer_id = 'h.customer_id)
      :- 'SubqueryAlias c
      :  +- 'UnresolvedRelation [customers], [], false
      +- 'SubqueryAlias h
         +- 'SubqueryAlias high_value
            +- 'SubqueryAlias high_value
               +- 'Project ['customer_id, 'total_spent, 'order_count]
                  +- 'Filter ('total_spent > 500)
                     +- 'SubqueryAlias customer_spend
                        +- 'SubqueryAlias customer_spend
    

In [6]:
# Window functions in SQL
window_sql = spark.sql("""
    SELECT
        salesperson,
        region,
        month,
        revenue,
        RANK() OVER (PARTITION BY salesperson ORDER BY revenue DESC) AS revenue_rank,
        SUM(revenue) OVER (PARTITION BY salesperson ORDER BY 
            CASE month
                WHEN 'January' THEN 1 WHEN 'February' THEN 2
                WHEN 'March'   THEN 3 WHEN 'April'    THEN 4
                WHEN 'May'     THEN 5 ELSE 6
            END
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW) AS running_total,
        LAG(revenue, 1) OVER (PARTITION BY salesperson ORDER BY 
            CASE month
                WHEN 'January' THEN 1 WHEN 'February' THEN 2
                WHEN 'March'   THEN 3 WHEN 'April'    THEN 4
                WHEN 'May'     THEN 5 ELSE 6
            END
        ) AS prev_revenue
    FROM sales
    WHERE salesperson = 'Alice'
    ORDER BY revenue_rank
""")

print("Window functions in SQL (Alice's data):")
window_sql.show()

{"ts": "2026-06-13 15:38:42.035", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `sales` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o30.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `sales` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXI

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `sales` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 22 pos 9;
'Sort ['revenue_rank ASC NULLS FIRST], true
+- 'Project ['salesperson, 'region, 'month, 'revenue, rank() windowspecdefinition('salesperson, 'revenue DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS revenue_rank#7, 'SUM('revenue) windowspecdefinition('salesperson, CASE WHEN ('month = January) THEN 1 WHEN ('month = February) THEN 2 WHEN ('month = March) THEN 3 WHEN ('month = April) THEN 4 WHEN ('month = May) THEN 5 ELSE 6 END ASC NULLS FIRST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS running_total#8, 'LAG('revenue, 1) wind

In [7]:
# Subqueries — correlated and non-correlated
subquery_result = spark.sql("""
    SELECT name, country, tier
    FROM customers
    WHERE customer_id IN (
        SELECT DISTINCT customer_id
        FROM orders
        WHERE amount > 500
    )
    ORDER BY name
""")
print("Customers with at least one order > 500 (subquery):")
subquery_result.show()

# Scalar subquery
scalar_result = spark.sql("""
    SELECT
        order_id,
        amount,
        ROUND(amount / (SELECT AVG(amount) FROM orders), 2) AS pct_of_avg
    FROM orders
    ORDER BY pct_of_avg DESC
    LIMIT 5
""")
print("Top orders as % of average (scalar subquery):")
scalar_result.show()

{"ts": "2026-06-13 15:38:42.246", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `customers` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o30.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `customers` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABL

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `customers` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 3 pos 9;
'Sort ['name ASC NULLS FIRST], true
+- 'Project ['name, 'country, 'tier]
   +- 'Filter 'customer_id IN (list#12 [])
      :  +- 'Distinct
      :     +- 'Project ['customer_id]
      :        +- 'Filter ('amount > 500)
      :           +- 'UnresolvedRelation [orders], [], false
      +- 'UnresolvedRelation [customers], [], false


In [8]:
# PIVOT in Spark SQL
# Rotate rows into columns — classic reporting pattern

pivot_result = spark.sql("""
    SELECT *
    FROM (
        SELECT salesperson, month, revenue
        FROM sales
        WHERE month IN ('January', 'February', 'March')
    )
    PIVOT (
        SUM(revenue)
        FOR month IN ('January' AS jan, 'February' AS feb, 'March' AS mar)
    )
    ORDER BY salesperson
""")

print("PIVOT — months as columns:")
pivot_result.show()

# Same thing in DataFrame API
print("Same using DataFrame .pivot():")
sales.filter(F.col("month").isin("January", "February", "March")) \
     .groupBy("salesperson") \
     .pivot("month", ["January", "February", "March"]) \
     .agg(F.sum("revenue")) \
     .orderBy("salesperson") \
     .show()

{"ts": "2026-06-13 15:38:42.502", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `sales` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o30.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `sales` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXI

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `sales` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 5 pos 13;
'Sort ['salesperson ASC NULLS FIRST], true
+- 'Project [*]
   +- 'Pivot 'month, [January, February, March], ['SUM('revenue)]
      +- 'SubqueryAlias __auto_generated_subquery_name
         +- 'Project ['salesperson, 'month, 'revenue]
            +- 'Filter 'month IN (January,February,March)
               +- 'UnresolvedRelation [sales], [], false


In [9]:
# ROLLUP and CUBE — hierarchical aggregations
rollup_result = spark.sql("""
    SELECT
        COALESCE(region, 'ALL REGIONS') AS region,
        COALESCE(salesperson, 'ALL SALESPEOPLE') AS salesperson,
        SUM(revenue) AS total_revenue
    FROM sales
    GROUP BY ROLLUP(region, salesperson)
    ORDER BY region, salesperson
""")

print("ROLLUP — subtotals per region and grand total:")
rollup_result.show()

{"ts": "2026-06-13 15:38:42.674", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `sales` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o30.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `sales` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXI

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `sales` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 6 pos 9;
'Sort ['region ASC NULLS FIRST, 'salesperson ASC NULLS FIRST], true
+- 'Aggregate [rollup(Vector(0), Vector(1), 'region, 'salesperson)], ['COALESCE('region, ALL REGIONS) AS region#16, 'COALESCE('salesperson, ALL SALESPEOPLE) AS salesperson#17, 'SUM('revenue) AS total_revenue#18]
   +- 'UnresolvedRelation [sales], [], false


## Part 3: Global Temp Views

In [10]:
# Global temp view — accessible from any SparkSession
orders.createOrReplaceGlobalTempView("global_orders")

# Access with global_temp prefix
spark.sql("SELECT COUNT(*) FROM global_temp.global_orders").show()

# Create a second SparkSession and access the global view
spark2 = SparkSession.builder \
    .appName("Session2") \
    .master("local[2]") \
    .getOrCreate()

print("Accessing global view from a different SparkSession:")
spark2.sql("SELECT COUNT(*) as count FROM global_temp.global_orders").show()

# Temp view is NOT visible from spark2
try:
    spark2.sql("SELECT COUNT(*) FROM orders")
except Exception as e:
    print(f"Temp view 'orders' NOT visible from spark2: {type(e).__name__}")

NameError: name 'orders' is not defined

## Part 4: Catalog API — Inspecting the Spark Metastore

In [11]:
# spark.catalog — inspect registered tables, databases, columns

print("Current database:", spark.catalog.currentDatabase())
print()

print("All registered tables:")
for t in spark.catalog.listTables():
    print(f"  {t.name:20s} | type={t.tableType:20s} | isTemp={t.isTemporary}")
print()

print("Columns of 'orders' view:")
for col in spark.catalog.listColumns("orders"):
    print(f"  {col.name:15s} | type={col.dataType}")

Current database: default

All registered tables:



Columns of 'orders' view:


AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `orders` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01;
'UnresolvedTableOrView [orders], Catalog.listColumns, true


In [12]:
# Manage views

# Check if a view exists before dropping
print(f"'orders' view exists: {spark.catalog.tableExists('orders')}")
print(f"'nonexistent' exists: {spark.catalog.tableExists('nonexistent')}")

# Drop a temp view
spark.catalog.dropTempView("customers")
print(f"'customers' after drop: {spark.catalog.tableExists('customers')}")

# Re-register it
customers.createOrReplaceTempView("customers")
print(f"'customers' after re-register: {spark.catalog.tableExists('customers')}")

'orders' view exists: False
'nonexistent' exists: False
'customers' after drop: False


NameError: name 'customers' is not defined

## Exercises

1. Write a SQL CTE that:
   - First computes total revenue per product category
   - Then ranks categories by revenue
   - Shows only top 2 categories

2. Write the same query as Exercise 1 using only the DataFrame API.

3. Use PIVOT in SQL to show total revenue per salesperson per month (columns = months Jan-Jun).

4. Register a global temp view for the `products` table. Create a new SparkSession and query it.

5. Using `spark.catalog`, list all tables and their types. Then drop and re-register the `sales` view.

In [13]:
# Exercise 1: CTE for top 2 categories
spark.sql("""
    WITH cat_revenue AS (
        SELECT p.category, SUM(o.amount) AS revenue
        FROM orders o
        JOIN products p ON o.product_id = p.product_id
        GROUP BY p.category
    ),
    ranked AS (
        SELECT *, RANK() OVER (ORDER BY revenue DESC) AS rnk
        FROM cat_revenue
    )
    SELECT category, revenue, rnk
    FROM ranked
    WHERE rnk <= 2
""").show()

{"ts": "2026-06-13 15:38:43.703", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `orders` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o30.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `orders` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF E

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `orders` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 4 pos 13;
'Project ['category, 'revenue, 'rnk]
+- 'Filter ('rnk <= 2)
   +- 'SubqueryAlias ranked
      +- 'SubqueryAlias ranked
         +- 'Project [*, rank() windowspecdefinition('revenue DESC NULLS LAST, specifiedwindowframe(RowFrame, unboundedpreceding$(), currentrow$())) AS rnk#42]
            +- 'SubqueryAlias cat_revenue
               +- 'SubqueryAlias cat_revenue
                  +- 'Aggregate ['p.category], ['p.category, 'SUM('o.amount) AS revenue#41]
                     +- 'Join Inner, ('o.product_id = 'p.product_id)
                        :- 'SubqueryAlias o
                        : 

In [14]:
# Exercise 3: PIVOT — revenue per salesperson per month
spark.sql("""
    SELECT *
    FROM (
        SELECT salesperson, month, revenue FROM sales
    )
    PIVOT (
        SUM(revenue)
        FOR month IN (
            'January' AS jan, 'February' AS feb, 'March' AS mar,
            'April' AS apr, 'May' AS may, 'June' AS jun
        )
    )
    ORDER BY salesperson
""").show()

{"ts": "2026-06-13 15:38:43.876", "level": "ERROR", "logger": "SQLQueryContextLogger", "msg": "[TABLE_OR_VIEW_NOT_FOUND] The table or view `sales` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01", "context": {"errorClass": "TABLE_OR_VIEW_NOT_FOUND"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o30.sql.\n: org.apache.spark.sql.AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `sales` cannot be found. Verify the spelling and correctness of the schema and catalog.\nIf you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.\nTo tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXI

AnalysisException: [TABLE_OR_VIEW_NOT_FOUND] The table or view `sales` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS. SQLSTATE: 42P01; line 4 pos 48;
'Sort ['salesperson ASC NULLS FIRST], true
+- 'Project [*]
   +- 'Pivot 'month, [January, February, March, April, May, June], ['SUM('revenue)]
      +- 'SubqueryAlias __auto_generated_subquery_name
         +- 'Project ['salesperson, 'month, 'revenue]
            +- 'UnresolvedRelation [sales], [], false
